# Query-response diagnostic

Run these cells from top to bottom to isolate where `Window()` fails.

This notebook checks:

1. runtime and secure-context detection
2. a direct IndexedDB roundtrip in the browser
3. `load_query_response_from_indexeddb(...)`
4. `Window(...)`


In [ ]:
%pip install hscmap

In [ ]:
import gc
import json
import sys
import traceback
import warnings

import js
from hscmap.comm import jupyterlab as jl
from pyodide.ffi import create_once_callable, run_sync


def print_block(title, value):
    print(f'[{title}]')
    if isinstance(value, str):
        print(value)
    else:
        print(json.dumps(value, indent=2, ensure_ascii=False))
    print()


def make_db_open_request():
    request = js.indexedDB.open(jl.query_response_indexeddb_name, 1)
    request.onupgradeneeded = create_once_callable(
        lambda _event: (
            request.result.createObjectStore(jl.query_response_indexeddb_store)
            if not request.result.objectStoreNames.contains(jl.query_response_indexeddb_store)
            else None
        )
    )
    return request


env_info = {
    'sys.platform': sys.platform,
    'browser_isSecureContext': bool(getattr(js, 'isSecureContext', False)),
    'python_is_secure_context': jl.is_secure_context(),
    'worker_location_origin': str(js.location.origin),
    'worker_location_href': str(js.location.href),
    'user_agent': str(js.navigator.userAgent),
    'query_response_storage_mode': (
        'indexeddb'
        if sys.platform == 'emscripten' and not jl.is_secure_context()
        else 'file-or-comm-cache'
    ),
    'candidate_dirs': [str(path) for path in jl.candidate_dirs('~query-probe')],
}
print_block('environment', env_info)

In [ ]:
js_query_id = f'js-roundtrip-{int(js.Date.now())}'
js_payload = json.dumps({'source': 'js-roundtrip', 'ok': True})

db = await jl.request_as_promise(make_db_open_request())
write_tx = db.transaction(jl.query_response_indexeddb_store, 'readwrite')
write_store = write_tx.objectStore(jl.query_response_indexeddb_store)
await jl.request_as_promise(write_store.put(js_payload, js_query_id))
await jl.transaction_as_promise(write_tx)

read_tx = db.transaction(jl.query_response_indexeddb_store, 'readonly')
read_store = read_tx.objectStore(jl.query_response_indexeddb_store)
js_roundtrip_value = await jl.request_as_promise(read_store.get(js_query_id))
await jl.transaction_as_promise(read_tx)

delete_tx = db.transaction(jl.query_response_indexeddb_store, 'readwrite')
delete_store = delete_tx.objectStore(jl.query_response_indexeddb_store)
await jl.request_as_promise(delete_store.delete(js_query_id))
await jl.transaction_as_promise(delete_tx)
db.close()

print(f'indexeddb_js_roundtrip_ok={js_roundtrip_value == js_payload}')
print(f'indexeddb_js_roundtrip_value={js_roundtrip_value!r}')

In [ ]:
async def run_sync_probe():
    return 'run_sync-ok'


with warnings.catch_warnings(record=True) as run_sync_records:
    warnings.simplefilter('always')
    run_sync_result = None
    run_sync_exception = None
    try:
        run_sync_result = run_sync(run_sync_probe())
    except Exception as exc:
        run_sync_exception = f'{type(exc).__name__}: {exc}'
    gc.collect()

print(f'run_sync_probe_result={run_sync_result!r}')
print(f'run_sync_probe_exception={run_sync_exception!r}')
print(f'run_sync_probe_warning_count={len(run_sync_records)}')
for index, record in enumerate(run_sync_records, start=1):
    print(f'run_sync_probe_warning_{index}={record.message}')

In [ ]:
helper_query_id = f'helper-roundtrip-{int(js.Date.now())}'
helper_payload = json.dumps({'source': 'python-helper', 'ok': True})

db = await jl.request_as_promise(make_db_open_request())
write_tx = db.transaction(jl.query_response_indexeddb_store, 'readwrite')
write_store = write_tx.objectStore(jl.query_response_indexeddb_store)
await jl.request_as_promise(write_store.put(helper_payload, helper_query_id))
await jl.transaction_as_promise(write_tx)
db.close()

with warnings.catch_warnings(record=True) as helper_records:
    warnings.simplefilter('always')
    helper_value = None
    helper_exception = None
    try:
        helper_value = jl.load_query_response_from_indexeddb(helper_query_id)
    except Exception as exc:
        helper_exception = f'{type(exc).__name__}: {exc}'
    gc.collect()

db = await jl.request_as_promise(make_db_open_request())
read_tx = db.transaction(jl.query_response_indexeddb_store, 'readonly')
read_store = read_tx.objectStore(jl.query_response_indexeddb_store)
helper_remaining_value = await jl.request_as_promise(read_store.get(helper_query_id))
await jl.transaction_as_promise(read_tx)
if helper_remaining_value is not None:
    delete_tx = db.transaction(jl.query_response_indexeddb_store, 'readwrite')
    delete_store = delete_tx.objectStore(jl.query_response_indexeddb_store)
    await jl.request_as_promise(delete_store.delete(helper_query_id))
    await jl.transaction_as_promise(delete_tx)
db.close()

print(f'indexeddb_helper_ok={helper_value == helper_payload}')
print(f'indexeddb_helper_value={helper_value!r}')
print(f'indexeddb_helper_exception={helper_exception!r}')
print(f'indexeddb_helper_remaining_value={helper_remaining_value!r}')
print(f'indexeddb_helper_warning_count={len(helper_records)}')
for index, record in enumerate(helper_records, start=1):
    print(f'indexeddb_helper_warning_{index}={record.message}')

In [ ]:
import time

from hscmap import Window


started_at = time.time()
with warnings.catch_warnings(record=True) as window_records:
    warnings.simplefilter('always')
    window_opened = False
    window_exception = None
    window_traceback = None
    window_title = None
    try:
        w = Window(title='query-response diagnostic')
        window_opened = True
        window_title = w.title
    except Exception as exc:
        window_exception = f'{type(exc).__name__}: {exc}'
        window_traceback = traceback.format_exc()
    gc.collect()

elapsed = time.time() - started_at
print(f'window_opened={window_opened}')
print(f'window_title={window_title!r}')
print(f'window_elapsed_seconds={elapsed:.2f}')
print(f'window_exception={window_exception!r}')
if window_traceback is not None:
    print(window_traceback)
print(f'window_warning_count={len(window_records)}')
for index, record in enumerate(window_records, start=1):
    print(f'window_warning_{index}={record.message}')